### This code is to implement SCD Type 2

In [0]:
# Assign libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# We are dealing with Patient Data. This is a existing patient data
existing_data = [
    (1, "John", "Doe", "M", 30, "2025-01-01", None, "Y"),
    (2, "Mary", "James", "F", 25, "2025-01-01", None, "Y")
]

existing_schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("is_current", StringType(), True)
])

dim_df = spark.createDataFrame(existing_data, existing_schema)

dim_df.show()

+----------+----------+---------+------+---+----------+--------+----------+
|patient_id|first_name|last_name|gender|age|start_date|end_date|is_current|
+----------+----------+---------+------+---+----------+--------+----------+
|         1|      John|      Doe|     M| 30|2025-01-01|    NULL|         Y|
|         2|      Mary|    James|     F| 25|2025-01-01|    NULL|         Y|
+----------+----------+---------+------+---+----------+--------+----------+



In [0]:
#Incoming source data

new_data = [
    (1, "John", "Doe", "M", 31),
    (2, "Mary", "James", "F", 25),
    (3, "Sam", "Wilson", "M", 40)
]

new_schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True)])

source_df = spark.createDataFrame(new_data, new_schema)

source_df.show()

+----------+----------+---------+------+---+
|patient_id|first_name|last_name|gender|age|
+----------+----------+---------+------+---+
|         1|      John|      Doe|     M| 31|
|         2|      Mary|    James|     F| 25|
|         3|       Sam|   Wilson|     M| 40|
+----------+----------+---------+------+---+



In [0]:
# Joined source table with dim table
joined_df = source_df.alias("source").join(dim_df.alias("dim"), "patient_id", "left")

# find out the new records
new_records_df = joined_df.filter(col("dim.patient_id").isNull()).select("source.*").\
    withColumn("start_date", lit(current_date().cast("string"))).\
        withColumn("end_date", lit(None).cast("string")).\
            withColumn("is_current", lit("Y"))

new_records_df.show()

new_records_df.printSchema()


+----------+----------+---------+------+---+----------+--------+----------+
|patient_id|first_name|last_name|gender|age|start_date|end_date|is_current|
+----------+----------+---------+------+---+----------+--------+----------+
|         3|       Sam|   Wilson|     M| 40|2026-05-14|    NULL|         Y|
+----------+----------+---------+------+---+----------+--------+----------+

root
 |-- patient_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- start_date: string (nullable = false)
 |-- end_date: string (nullable = true)
 |-- is_current: string (nullable = false)



In [0]:
# New version Records
new_version_df = joined_df.filter((col("source.first_name") != col("dim.first_name")) | \
    (col("source.last_name") != col("dim.last_name")) | \
    (col("source.gender") != col("dim.gender")) | \
    (col("source.age") != col("dim.age"))).select("source.*").\
        withColumn("start_date", lit(current_date().cast("string"))).\
            withColumn("end_date", lit(None).cast("string")).\
                withColumn("is_current", lit("Y"))

new_version_df.show()

+----------+----------+---------+------+---+----------+--------+----------+
|patient_id|first_name|last_name|gender|age|start_date|end_date|is_current|
+----------+----------+---------+------+---+----------+--------+----------+
|         1|      John|      Doe|     M| 31|2026-05-14|    NULL|         Y|
+----------+----------+---------+------+---+----------+--------+----------+



In [0]:
# Expired records
expired_df = joined_df.filter((col("source.first_name") != col("dim.first_name")) | \
    (col("source.last_name") != col("dim.last_name")) | \
    (col("source.gender") != col("dim.gender")) | \
    (col("source.age") != col("dim.age"))).select("dim.*").\
            withColumn("end_date", lit((current_date()-1).cast("string"))).\
                withColumn("is_current", lit("N"))

expired_df.show() 

+----------+----------+---------+------+---+----------+----------+----------+
|patient_id|first_name|last_name|gender|age|start_date|  end_date|is_current|
+----------+----------+---------+------+---+----------+----------+----------+
|         1|      John|      Doe|     M| 30|2025-01-01|2026-05-13|         N|
+----------+----------+---------+------+---+----------+----------+----------+



In [0]:
# Merge all datasets
df_merged = new_records_df.union(new_version_df).union(expired_df)
df_merged.show()

#Final
df_final = df_merged.union(dim_df.join(df_merged, "patient_id", "left_anti"))

df_final.show()     

+----------+----------+---------+------+---+----------+----------+----------+
|patient_id|first_name|last_name|gender|age|start_date|  end_date|is_current|
+----------+----------+---------+------+---+----------+----------+----------+
|         3|       Sam|   Wilson|     M| 40|2026-05-14|      NULL|         Y|
|         1|      John|      Doe|     M| 31|2026-05-14|      NULL|         Y|
|         1|      John|      Doe|     M| 30|2025-01-01|2026-05-13|         N|
+----------+----------+---------+------+---+----------+----------+----------+

+----------+----------+---------+------+---+----------+----------+----------+
|patient_id|first_name|last_name|gender|age|start_date|  end_date|is_current|
+----------+----------+---------+------+---+----------+----------+----------+
|         3|       Sam|   Wilson|     M| 40|2026-05-14|      NULL|         Y|
|         1|      John|      Doe|     M| 31|2026-05-14|      NULL|         Y|
|         1|      John|      Doe|     M| 30|2025-01-01|2026-05-